# 04 - Robustness Checks

Notebook goals:
- Run pre-defined sensitivity scenarios
- Compare key metrics across scenarios
- Inspect IRF path differences visually

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from nasdaq_svar.presentation import build_chapter4_conclusion
from nasdaq_svar.sensitivity import run_sensitivity

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
result = run_sensitivity(
    plan_path=PROJECT_ROOT / "configs/sensitivity.yaml",
    project_root=PROJECT_ROOT,
)
result

In [ ]:
summary_path = Path(result["summary_csv"])
summary = pd.read_csv(summary_path)
summary = summary.sort_values("irf_trough_value")
summary

In [ ]:
irf_paths = summary_path.parent / "sensitivity_irf_paths.csv"
irf = pd.read_csv(irf_paths)

fig, ax = plt.subplots(figsize=(11, 5))
for scenario, grp in irf.groupby("scenario"):
    ax.plot(grp["horizon"], grp["orth_irf"], lw=1.8, label=scenario)
ax.axhline(0.0, color="black", lw=1.0, linestyle="--")
ax.set_title("IRF sensitivity comparison")
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("NASDAQ_SA response")
ax.legend(frameon=False, ncols=2)
fig.tight_layout()

In [ ]:
summary[[
    "scenario",
    "selected_lag",
    "irf_impact_h0",
    "irf_trough_value",
    "irf_trough_horizon",
    "fevd_policy_share_h12",
    "fevd_policy_share_h24",
]]

## Slide-Ready Conclusion

In [ ]:
chapter4_text = build_chapter4_conclusion(summary)
display(Markdown(chapter4_text))